# Delta Lake MERGE Implementation
### Objective
Perform incremental data processing using **Delta Lake**.

**Steps covered in this notebook:**
1. Load dataset into a Delta table
2. Perform basic cleaning (handle nulls, remove duplicates)
3. Create a second dataset simulating new/incremental data
4. Apply `MERGE` operation to update existing and insert new records
5. Validate results (row count, duplicates)
6. Display final dataset and summary


## 0. Environment Setup
Install and configure PySpark with Delta Lake support.

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install pyspark==3.5.0 delta-spark==3.1.0

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pyspark.sql.functions as F

builder = (
    SparkSession.builder.appName("DeltaLakeMergeAssignment")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


## 1. Load Dataset into a Delta Table
We start with a `customer_master.csv` file representing our base/master dataset.

In [ ]:
# Path where the Delta table will be stored
delta_table_path = "data/delta/customer_master"
source_csv_path = "data/customer_master.csv"

# --- If you don't already have a source CSV, generate a sample one ---
sample_master_data = [
    (1, "Alice",   "alice@example.com",   "Jaipur",  None),
    (2, "Bob",     "bob@example.com",     "Mumbai",  "Gold"),
    (3, "Charlie", None,                  "Delhi",   "Silver"),
    (4, "David",   "david@example.com",   "Chennai", "Gold"),
    (5, "Eva",     "eva@example.com",     "Jaipur",  "Silver"),
    (5, "Eva",     "eva@example.com",     "Jaipur",  "Silver"),  # duplicate row
]
columns = ["customer_id", "name", "email", "city", "tier"]

df_master_raw = spark.createDataFrame(sample_master_data, columns)
df_master_raw.write.mode("overwrite").option("header", True).csv(source_csv_path)

# Load the CSV into a DataFrame
df_master = spark.read.option("header", True).csv(source_csv_path)
df_master.show(truncate=False)


In [ ]:
# Write the raw data into a Delta table for the first time
df_master.write.format("delta").mode("overwrite").save(delta_table_path)

# Register it as a SQL table for convenience (optional)
spark.sql(f"CREATE TABLE IF NOT EXISTS customer_master USING DELTA LOCATION '{delta_table_path}'")

print("Delta table created at:", delta_table_path)
spark.read.format("delta").load(delta_table_path).show(truncate=False)


## 2. Basic Cleaning
Handle null values and remove duplicate rows before treating this as our clean baseline dataset.

In [ ]:
df_clean = spark.read.format("delta").load(delta_table_path)

print("Row count before cleaning:", df_clean.count())

# Drop exact duplicate rows
df_clean = df_clean.dropDuplicates()

# Handle nulls: fill missing email/tier with placeholders instead of dropping rows
df_clean = df_clean.fillna({"email": "unknown@example.com", "tier": "Standard"})

print("Row count after cleaning:", df_clean.count())
df_clean.show(truncate=False)

# Overwrite the Delta table with the cleaned version
df_clean.write.format("delta").mode("overwrite").save(delta_table_path)


## 3. Create Incremental Dataset
Simulate a `customer_incremental.csv` file containing:
- **Updates** to existing customers (e.g., changed city/tier)
- **New** customers not present in the master table

In [ ]:
incremental_data = [
    (2, "Bob",     "bob@example.com",     "Pune",     "Platinum"),  # existing -> update (city & tier changed)
    (4, "David",   "david.new@example.com", "Chennai", "Gold"),      # existing -> update (email changed)
    (6, "Farah",   "farah@example.com",   "Bengaluru", "Gold"),      # new customer -> insert
    (7, "Gopal",   "gopal@example.com",   "Kolkata",  "Standard"),   # new customer -> insert
]

df_incremental = spark.createDataFrame(incremental_data, columns)

incremental_csv_path = "data/customer_incremental.csv"
df_incremental.write.mode("overwrite").option("header", True).csv(incremental_csv_path)

df_incremental.show(truncate=False)


## 4. Apply MERGE Operation
Use Delta Lake's `MERGE INTO` to:
- **UPDATE** matching records (based on `customer_id`)
- **INSERT** new records that don't exist in the target table

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_table_path)

(
    delta_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "name":  "source.name",
        "email": "source.email",
        "city":  "source.city",
        "tier":  "source.tier",
    })
    .whenNotMatchedInsert(values={
        "customer_id": "source.customer_id",
        "name":        "source.name",
        "email":       "source.email",
        "city":        "source.city",
        "tier":        "source.tier",
    })
    .execute()
)

print("MERGE operation completed successfully.")


## 5. Validate Results
Check row counts and confirm there are no duplicate `customer_id`s after the merge.

In [ ]:
df_after_merge = spark.read.format("delta").load(delta_table_path)

total_rows = df_after_merge.count()
distinct_ids = df_after_merge.select("customer_id").distinct().count()

print(f"Total rows after MERGE : {total_rows}")
print(f"Distinct customer_ids  : {distinct_ids}")
print("Duplicates present:", total_rows != distinct_ids)

# Show duplicate customer_ids, if any
df_after_merge.groupBy("customer_id").count().filter("count > 1").show()


## 6. Display Final Dataset and Summary

In [ ]:
print("Final Delta Table Contents:")
df_after_merge.orderBy("customer_id").show(truncate=False)

print("=== Summary ===")
print(f"Rows in original master dataset (after cleaning): {df_clean.count()}")
print(f"Rows in incremental dataset:                       {df_incremental.count()}")
print(f"Rows in final merged Delta table:                  {total_rows}")
print(f"Updated existing records:                          2 (customer_id 2, 4)")
print(f"Newly inserted records:                            2 (customer_id 6, 7)")


## 7. Delta Lake Time Travel (Bonus)
Delta Lake automatically versions every write, so we can inspect table history and even query previous versions.

In [ ]:
delta_table.history().select("version", "timestamp", "operation").show(truncate=False)

# Example: read the table as it was right after step 1 (version 0)
# spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path).show()


## Conclusion
This notebook demonstrated a complete incremental data processing workflow using Delta Lake:
loading data into a Delta table, cleaning it, simulating incremental updates, applying a `MERGE`
operation to upsert records, and validating the final result. Delta Lake's transaction log also
gives us built-in versioning/time-travel for free, which is useful for auditing incremental loads.